# Day 1, Hands-on. Your first encoder run

**This is the participant notebook.** Runs on free Colab CPU in under 3 minutes.

The goal is to know that you can. By the end, you have run a state-of-the-art encoder model on real US trade-policy text, on free hardware, from your own browser.

## How to use this

1. **File · Save a copy in Drive** so you keep your edits.
2. **Runtime · Run all** to run every cell from the top.
3. The first cell installs the library. Wait for it to finish (about 30 seconds).
4. The second cell downloads a model. Wait for it to finish (about 45 seconds on first run).
5. After that, each classification takes a second or two.

## Setup

One install, one import. That is the whole environment.

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import pipeline

MODEL_ID = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'
zsc = pipeline('zero-shot-classification', model=MODEL_ID)
print('Model loaded. Ready.')

## Five real excerpts from US National Trade Estimate reports

Each is a sentence or two from a country chapter on intellectual-property rights. You can imagine each as a small piece of the kind of paragraph Park's paper analyzes.

In [ ]:
excerpts = [
    ('PERU 2015',
     'Peru was listed on the Watch List in the 2013 Special 301 Report. '
     'Pirated and counterfeit goods remain widely available in Peru.'),

    ('JORDAN 2016',
     'Jordan was not listed in the 2015 Special 301 Report. The Jordanian government continues '
     'to take steps to provide more comprehensive protection of intellectual property rights.'),

    ('BAHRAIN 2005',
     'Bahrain is in the process of joining the WIPO Copyright Treaty. '
     'The government has made significant progress in reducing copyright piracy.'),

    ('TURKEY 2017',
     'Turkey remained on the Watch List in the 2016 Special 301 Report. Turkish law enforcement '
     'efforts to improve IPR enforcement have decreased in the past year.'),

    ('ISRAEL 2017',
     'The United States removed Israel from the Special 301 Report in 2014. '
     'The United States remains concerned with deficiencies in Israel protections for copyrighted works.'),
]

for tag, text in excerpts:
    print(f'[{tag}]')
    print(text)
    print()

## Experiment 1. Run as-is

We give the model three candidate labels and ask which best fits each excerpt. The model returns a probability for each label. The label with the highest probability is the prediction.

In [ ]:
candidate_labels = [
    'the country has weak IP protection and the US criticizes it',
    'the country has strong IP protection and the US praises it',
    'unrelated to IP protection',
]

for tag, text in excerpts:
    out = zsc(text, candidate_labels=candidate_labels)
    top_label = out['labels'][0]
    top_score = out['scores'][0]
    print(f'[{tag}] -> {top_label}  ({top_score:.2f})')

**What to notice.**

- Easy paragraphs (Peru, Bahrain) get classified the way you would expect.
- Mixed paragraphs (Israel) often surface the surprising classification, because the model picks the dominant signal in the text rather than weighing both halves.
- Day 2 fixes this with hypothesis engineering and validation against a hand-coded gold set.

## Experiment 2. Change the candidate labels

The same text plus different labels gives different predictions. The model is not reading the labels as keywords. It is checking whether the text *entails* each label as a hypothesis. Wording matters.

Try editing the `candidate_labels` list below. Some ideas to try.
- Make the labels shorter (e.g., "weak IP", "strong IP", "other").
- Add a fourth label.
- Phrase one label as a question.

In [ ]:
your_labels = [
    'IP enforcement is weak',
    'IP enforcement is improving',
    'IP enforcement is strong',
]

for tag, text in excerpts:
    out = zsc(text, candidate_labels=your_labels)
    top_label = out['labels'][0]
    top_score = out['scores'][0]
    print(f'[{tag}] -> {top_label}  ({top_score:.2f})')

## Experiment 3. Your own text

Replace the text below with anything from your own research. The model has never seen your data. Often it still produces a useful first cut.

In [ ]:
your_text = (
    'Replace this string with a sentence or paragraph from your own research. '
    'It can be a tweet, a speech excerpt, a press release. Anything in English.'
)

your_labels = [
    'positive',
    'negative',
    'neutral',
]

out = zsc(your_text, candidate_labels=your_labels)
for lbl, sc in zip(out['labels'], out['scores']):
    print(f'  {sc:0.3f}  {lbl}')

## What you just did

You loaded a 184-million-parameter Transformer model on Google's servers from your browser, in three lines of Python, with no API key, for free. You then used it to classify five real US trade-report excerpts and tested how the labels you choose change the answer.

If you got here, you can do this on your own data later.

**Day 2** turns this into a publishable measurement workflow with hand-coded gold sets, hypothesis engineering, and validation that closes the off-the-shelf gap.